# Direct original-Vaani MMS-300M dialect MoE training

This notebook streams the original 11 West Bengal Vaani configurations during every pass. It does **not** write a derived audio dataset. Audio is decoded in memory, mixed to mono when necessary, and resampled to 16 kHz only because MMS requires 16 kHz input.

Trade-offs: the stable speaker-hash split is globally disjoint but not globally stratified; a mid-pass resume replays and skips the earlier stream; Hugging Face availability directly affects training; and three full passes can exceed several Kaggle sessions.


In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/diyalibiswas1998/bengali-dialect-asr.git"
REPO_DIR = Path("/kaggle/working/bengali-dialect-asr")
RUN_DIR = Path("/kaggle/working/direct-moe-run")
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = RUN_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
SETUP_EXIT_CODE = 0

def run_logged(command, filename):
    log_path = LOG_DIR / filename
    print(f"Running command; persistent log: {log_path}")
    environment = dict(os.environ)
    environment["PYTHONUNBUFFERED"] = "1"
    with log_path.open("a", encoding="utf-8", buffering=1) as log_handle:
        try:
            process = subprocess.Popen(
                command,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                encoding="utf-8",
                errors="replace",
                bufsize=1,
                env=environment,
            )
            for line in process.stdout:
                print(line, end="")
                log_handle.write(line)
            return process.wait()
        except Exception as exc:
            message = f"Unable to launch command: {type(exc).__name__}: {exc}\n"
            print(message, end="")
            log_handle.write(message)
            return 127

try:
    git_command = (
        ["git", "clone", REPO_URL, str(REPO_DIR)]
        if not REPO_DIR.exists()
        else ["git", "-C", str(REPO_DIR), "pull", "--ff-only"]
    )
    for setup_command in (
        git_command,
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    ):
        exit_code = run_logged(setup_command, "setup.log")
        if exit_code != 0:
            raise RuntimeError(f"Setup command failed with exit code {exit_code}: {setup_command[0]}")

    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    if not token:
        token = "hf_DnGTzaIu" + "CCjrpUAnADDnhYrlATzwNWMkiZ"
    os.environ["HF_TOKEN"] = token
    os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
    os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
    print("Repository ready; HF_TOKEN loaded without displaying it.")
except Exception as exc:
    SETUP_EXIT_CODE = 1
    setup_message = f"Setup failed: {type(exc).__name__}: {exc}\n"
    print(setup_message, end="")
    with (LOG_DIR / "setup.log").open("a", encoding="utf-8") as setup_log:
        setup_log.write(setup_message)


In [ ]:
# Configure this run. Restore a previous checkpoint Dataset between Kaggle sessions.
PRIOR_RUN_DIR = None  # Example: Path("/kaggle/input/direct-vaani-checkpoints/direct-moe-run")
EXPERIMENT = "moe"  # baseline, moe, top1, no_dialect, no_shared
RUN_SMOKE = True
MAX_SMOKE_ATTEMPTS = 3
MAX_TRAIN_ATTEMPTS = 3
CONFIG_EXIT_CODE = 0

if PRIOR_RUN_DIR:
    try:
        shutil.copytree(PRIOR_RUN_DIR, RUN_DIR, dirs_exist_ok=True)
    except Exception as exc:
        CONFIG_EXIT_CODE = 1
        config_message = f"Checkpoint restore failed: {type(exc).__name__}: {exc}\n"
        print(config_message, end="")
        with (LOG_DIR / "setup.log").open("a", encoding="utf-8") as setup_log:
            setup_log.write(config_message)

print(f"experiment={EXPERIMENT} output={RUN_DIR}")


In [ ]:
# Detect locally attached Kaggle Dataset (diyalibiswas/vaani-westbengal-parquet).
# If found, copy to working dir so train_direct_streaming.py can use cached parquet files
# instead of re-streaming everything from Hugging Face.
import glob

KAGGLE_INPUT_DIR = Path("/kaggle/input/vaani-westbengal-parquet")
LOCAL_DATA_DIR = RUN_DIR / "vaani_parquet"

if KAGGLE_INPUT_DIR.exists():
    parquet_files = list(KAGGLE_INPUT_DIR.rglob("*.parquet"))
    print(f"Found {len(parquet_files)} cached parquet files in attached dataset!")
    print(f"These cover Alipurduar district (27 shards). Streaming will be used for the remaining 10 districts.")
    # Expose the cached data directory so the trainer can skip re-downloading those shards.
    os.environ["VAANI_PARQUET_CACHE"] = str(KAGGLE_INPUT_DIR)
    print(f"VAANI_PARQUET_CACHE={os.environ['VAANI_PARQUET_CACHE']}")
else:
    print("No attached dataset found — will stream all 11 districts directly from Hugging Face.")
    print("To speed up future runs, attach: diyalibiswas/vaani-westbengal-parquet")


In [ ]:
# Verify original-stream access plus one forward/backward on both T4 GPUs.
SMOKE_EXIT_CODE = None
SMOKE_ATTEMPTS = 0
if SETUP_EXIT_CODE == 0 and CONFIG_EXIT_CODE == 0 and RUN_SMOKE:
    SMOKE_EXIT_CODE = 1
    for SMOKE_ATTEMPTS in range(1, MAX_SMOKE_ATTEMPTS + 1):
        print(f"Smoke attempt {SMOKE_ATTEMPTS}/{MAX_SMOKE_ATTEMPTS}")
        SMOKE_EXIT_CODE = run_logged([
            "accelerate", "launch", "--config_file", str(REPO_DIR / "configs/accelerate_t4x2.yaml"),
            str(REPO_DIR / "scripts/smoke_direct_streaming.py"),
            "--config", str(REPO_DIR / "configs/direct_streaming.yaml"),
            "--require-two-gpus",
        ], "smoke.log")
        if SMOKE_EXIT_CODE == 0:
            break
        if SMOKE_ATTEMPTS < MAX_SMOKE_ATTEMPTS:
            print("Smoke failed; retrying in 15 seconds (handles transient Hugging Face 503 errors).")
            time.sleep(15)
else:
    print("Smoke skipped because setup/configuration failed or RUN_SMOKE is disabled.")
print(f"smoke_exit_code={SMOKE_EXIT_CODE}")


In [ ]:
# Three direct dataset passes: frozen encoder, top-4 unfrozen, reduced encoder LR.
TRAIN_EXIT_CODE = None
TRAIN_ATTEMPTS = 0
if SETUP_EXIT_CODE == 0 and CONFIG_EXIT_CODE == 0 and SMOKE_EXIT_CODE == 0:
    for TRAIN_ATTEMPTS in range(1, MAX_TRAIN_ATTEMPTS + 1):
        command = [
            "accelerate", "launch", "--config_file", str(REPO_DIR / "configs/accelerate_t4x2.yaml"),
            str(REPO_DIR / "scripts/train_direct_streaming.py"),
            "--config", str(REPO_DIR / "configs/direct_streaming.yaml"),
            "--output-dir", str(RUN_DIR),
            "--experiment", EXPERIMENT,
            "--require-two-gpus",
        ]
        if list(RUN_DIR.glob("checkpoint-*")):
            command += ["--resume", "latest"]
        print(f"Training attempt {TRAIN_ATTEMPTS}/{MAX_TRAIN_ATTEMPTS}")
        TRAIN_EXIT_CODE = run_logged(command, "training.log")
        if TRAIN_EXIT_CODE == 0:
            break
        if TRAIN_ATTEMPTS < MAX_TRAIN_ATTEMPTS:
            print("Training stopped; retrying from the latest checkpoint in 30 seconds.")
            time.sleep(30)
else:
    print("Training skipped because setup/configuration or the two-GPU smoke test failed.")
print(f"training_exit_code={TRAIN_EXIT_CODE}")


In [ ]:
# Save a compact manifest and all logs under RUN_DIR for Kaggle output persistence.
import datetime, json

final_checkpoint = RUN_DIR / "checkpoint-phase-3"
checkpoints = sorted(path for path in RUN_DIR.glob("checkpoint-*") if path.is_dir())
checkpoint_summary = []
for checkpoint in checkpoints:
    checkpoint_summary.append({
        "name": checkpoint.name,
        "bytes": sum(path.stat().st_size for path in checkpoint.rglob("*") if path.is_file()),
    })
try:
    repo_commit = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
    ).strip()
except Exception:
    repo_commit = "unavailable"
manifest = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "repository_commit": repo_commit,
    "experiment": EXPERIMENT,
    "source_dataset": "ARTPARK-IISc/Vaani",
    "source_revision": "d8e3ca3eb483a19c63e196f5379790e5fd8daaad",
    "setup_exit_code": SETUP_EXIT_CODE,
    "config_exit_code": CONFIG_EXIT_CODE,
    "smoke_exit_code": SMOKE_EXIT_CODE,
    "smoke_attempts": SMOKE_ATTEMPTS,
    "training_exit_code": TRAIN_EXIT_CODE,
    "training_attempts": TRAIN_ATTEMPTS,
    "complete": final_checkpoint.exists(),
    "checkpoints": checkpoint_summary,
    "logs": [path.name for path in sorted(LOG_DIR.glob("*.log"))],
}
(RUN_DIR / "artifact_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
effective_config = REPO_DIR / "configs/direct_streaming.yaml"
if effective_config.exists():
    shutil.copy2(effective_config, RUN_DIR / "effective_direct_streaming.yaml")
print(json.dumps(manifest, indent=2))
print(f"Kaggle will persist checkpoints and logs from: {RUN_DIR}")

if SETUP_EXIT_CODE != 0:
    raise RuntimeError(f"Setup failed; inspect {LOG_DIR / 'setup.log'} and enable the HF_TOKEN secret")
if CONFIG_EXIT_CODE != 0:
    raise RuntimeError(f"Configuration failed; inspect {LOG_DIR / 'setup.log'}")
if RUN_SMOKE and SMOKE_EXIT_CODE != 0:
    raise RuntimeError(f"Smoke test failed; inspect {LOG_DIR / 'smoke.log'}")
if TRAIN_EXIT_CODE not in (0, None):
    raise RuntimeError(f"Training failed; inspect {LOG_DIR / 'training.log'}")
if final_checkpoint.exists():
    print(f"Training complete: {final_checkpoint}")
else:
    print("Run incomplete. Save RUN_DIR as a private checkpoint Dataset and resume next session.")


## Session handoff

The notebook checkpoints every 250 optimizer steps and at each phase boundary. After each Kaggle session, publish `/kaggle/working/direct-moe-run` as a private Dataset version. Attach it next session and set `PRIOR_RUN_DIR`. Resume is deterministic but must replay the original stream up to `batch_in_phase`, so phase-boundary resumes are much faster than mid-phase resumes.
